In [ ]:
%matplotlib ipympl
#import cProfile, pstats, io
#from pstats import SortKey
import warnings

#import scipy as sp
import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

In [ ]:
N = 3
a = qt.create(N).dag()
lindbladian = qt.lindblad_dissipator(a)
Id = qt.identity(N)
lindbladians = [qt.super_tensor(qt.to_super(Id),lindbladian),qt.super_tensor(lindbladian,qt.to_super(Id))]

In [ ]:
def QFIMatrix(rho, lindbladians):
    (lambdas, Uw) = rho.eigenstates(output_type='oper')
    dims = rho.dims
    lambdas = np.round(lambdas, 12)# just round off small values since they are often erroneous
    #lamfixed = np.where(lambdas>0,lambdas,0)
    U = qt.dimensions.from_tensor_rep(np.reshape(qt.dimensions.to_tensor_rep(Uw),np.array(dims).flatten()),dims)
    leffs = [U.dag() * qt.vector_to_operator(lindbladian*qt.operator_to_vector(rho))*U for lindbladian in lindbladians]
    [lambda1,lambda2] = np.meshgrid(lambdas,lambdas)

    # Vectorized logic to calculate the QFI
    # May be improved in the future, but works very well now
    matsize = [np.prod(dims[0]),np.prod(dims[1])]
    tops = [np.reshape(qt.dimensions.to_tensor_rep(leff),matsize) for leff in leffs]
    bot = (lambda1 + lambda2)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="invalid value encountered in divide")
        warnings.filterwarnings("ignore", message="divide by zero encountered in divide")
        FIs = [[ np.nansum((top1*np.transpose(top2)+np.transpose(top1)*top2)/bot) for top1 in tops] for top2 in tops]
    return np.real_if_close(FIs)

In [ ]:
def BMat(avals):
    size = np.size(avals)
    B = np.identity(size)
    ind = np.nonzero(avals)[0][0]
    mask = np.ones(size, dtype = bool)
    mask[ind] = 0
    B[0,ind] = 1
    B[ind,ind]=0
    B[ind,0] = 1/avals[ind]
    B[mask,0] = -avals[mask]/avals[ind]
    return B

In [ ]:
def Calc_Qeff(rho0, annihilators, weights, transmisivities):
    # "Hot Path" for this function is primarily the matrix exponentiation
    # If we want to reuse stuff it might be simplest to exponentiate once
    # and then repeatedly multiply them
    
    # First calculate the state after the beamsplitters
    dims = rho0.dims
    ident = qt.identity(dims[0])
    vac = qt.basis(dims[0]).proj()
    thetas = np.arccos(np.sqrt(transmisivities))
    As = [qt.tensor(a,ident).dag()*qt.tensor(ident,a) + qt.tensor(a,ident)*qt.tensor(ident,a).dag() for a in annihilators]
    Gs = thetas *1j*As
    U = np.sum(Gs).expm()
    rhoexpand = qt.tensor(rho0,vac)
    rhofexpand = U * rhoexpand*U.dag()
    rhof = rhofexpand.ptrace(np.arange(np.size(dims[0])))
    # Then create the Lindbladians
    lindbladians = [ qt.lindblad_dissipator(a) for a in annihilators]

    tans = np.tan(thetas)#np.sqrt((1-transmisivities)/transmisivities)

    # Create matricies
    QFI = QFIMatrix(rhof,lindbladians)
    D = np.diag(tans)

    B = BMat(weights)
    QFIT = B@D@QFI@D@np.transpose(B)
    Qeffi = np.linalg.pinv(QFIT,hermitian=True)[0,0]
    return 1/Qeffi

In [ ]:
rho0_1d = (qt.basis(N,2)+qt.basis(N)).unit().proj()
#res_1d = qt.mesolve(lindbladian,rho0_1d,np.linspace(0,.1,1000))

In [ ]:
rho0 = (qt.tensor(qt.basis(N),qt.basis(N,2)) + qt.tensor(qt.basis(N,2),qt.basis(N))).unit().proj() #qt.tensor(rho0_1d,rho0_1d)
#res = qt.mesolve(lindbladians,rho0,np.linspace(0,.1,1000))

In [ ]:
Calc_Qeff(rho0, [qt.tensor(a,Id),qt.tensor(Id,a)],np.array([1,1]),np.array([.1,.1]))

In [ ]:
etas = np.linspace(0.01,0.99,12)
[eta1,eta2] = np.meshgrid(etas,etas)
#with cProfile.Profile() as pr:
Qs = np.array([[ Calc_Qeff(rho0, [qt.tensor(a,Id),qt.tensor(Id,a)],np.array([1,.3]),np.array([eta1[i,j],eta2[i,j]])) for j in range(12)] for i in range(12)])
#eps = pstats.Stats(pr).sort_stats("cumtime")  # cumulative time by function
#ps.print_stats(20)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(eta1,eta2,Qs)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Effective Fisher Information')
plt.show()

In [ ]:
a0 = .7
a1 = np.sqrt(1-a0**2)
A = (a0*qt.tensor(a,Id) + a1*qt.tensor(Id,a))
vac = qt.basis([N,N,N])
rho = ((qt.tensor(A.dag()**2,Id) +qt.tensor(Id,Id,a.dag()))*vac).unit().proj()

In [ ]:
S = 6
etas = np.linspace(0.01,0.99,S)
[eta1,eta2] = np.meshgrid(etas,etas)
#with cProfile.Profile() as pr:
Qs = np.array([[ Calc_Qeff(rho, [qt.tensor(a,Id,Id),qt.tensor(Id,a,Id)],np.array([a0,a1]),np.array([eta1[i,j],eta2[i,j]])) for j in range(S)] for i in range(S)])
#eps = pstats.Stats(pr).sort_stats("cumtime")  # cumulative time by function
#ps.print_stats(20)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(eta1,eta2,Qs)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Effective Fisher Information')
plt.show()

In [ ]:
num_photons = .01

In [ ]:
r = np.arcsinh(np.sqrt(num_photons))
Sq = qt.squeezing(qt.tensor(A,Id),qt.tensor(Id,Id,a),r)
rho = (Sq*vac).unit().proj()

In [ ]:
S = 6
etas = np.linspace(0.01,0.99,S)
[eta1,eta2] = np.meshgrid(etas,etas)
#with cProfile.Profile() as pr:
Qs = np.array([[ Calc_Qeff(rho, [qt.tensor(a,Id,Id),qt.tensor(Id,a,Id)],np.array([a0,a1]),np.array([eta1[i,j],eta2[i,j]])) for j in range(S)] for i in range(S)])
#eps = pstats.Stats(pr).sort_stats("cumtime")  # cumulative time by function
#ps.print_stats(20)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(eta1,eta2,Qs)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Effective Fisher Information')
plt.show()

In [ ]:
alph = np.sqrt(num_photons)
rho = qt.tensor(qt.coherent_dm(N,alph*a0),qt.tensor(qt.coherent_dm(N,alph*a1)))

In [ ]:
S = 6
etas = np.linspace(0.01,0.99,S)
[eta1,eta2] = np.meshgrid(etas,etas)
#with cProfile.Profile() as pr:
Qs = np.array([[ Calc_Qeff(rho, [qt.tensor(a,Id),qt.tensor(Id,a)],np.array([a0,a1]),np.array([eta1[i,j],eta2[i,j]])) for j in range(S)] for i in range(S)])
#eps = pstats.Stats(pr).sort_stats("cumtime")  # cumulative time by function
#ps.print_stats(20)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(eta1,eta2,Qs)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Effective Fisher Information')
plt.show()